# NF Sweep V2 Quick Check

Use this notebook for a fast sanity check of the `nf_sweep_v2` models after training. It can generate **one raw sample per completed model** and then run the standard first diagnostics:

1. checkpoint/sample discovery
2. one-point statistics
3. P(k) comparison
4. generated image grid
5. training-curve status

The one-sample outputs are written to `results/nf_sweep_v2/quickcheck_samples/`, not the production sample directory. This avoids poisoning the production sampling array with 1-sample files.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import shlex
import subprocess
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", Path.cwd())).resolve()
if not (PROJECT_DIR / "scripts").exists():
    PROJECT_DIR = Path("/home/jiamingp/diffusion_models_repo")

COSMODIFF_DIR = Path(os.environ.get(
    "COSMODIFF_DIR_OVERRIDE",
    "/home/jiamingp/Diffusion_model/cosmo_diffusion_main",
)).resolve()
if COSMODIFF_DIR.exists() and str(COSMODIFF_DIR) not in sys.path:
    sys.path.insert(0, str(COSMODIFF_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180})
print("project:", PROJECT_DIR)
print("cosmodiff_dir:", COSMODIFF_DIR, "exists=", COSMODIFF_DIR.exists())

In [ ]:
SWEEP_NAME = "nf_sweep_v2"
CONFIG_DIR = PROJECT_DIR / "local" / SWEEP_NAME / "configs"
MANIFEST_PATH = PROJECT_DIR / "local" / SWEEP_NAME / "manifest.json"
CHECKPOINT_ROOT = Path("/scratch/huterer_root/huterer0/jiamingp/saved_runs/nf_sweep_v2")
QUICKCHECK_SAMPLE_ROOT = PROJECT_DIR / "results" / SWEEP_NAME / "quickcheck_samples"
PRODUCTION_SAMPLE_ROOT = PROJECT_DIR / "results" / SWEEP_NAME / "samples"
OUTPUT_DIR = PROJECT_DIR / "results" / SWEEP_NAME / "quickcheck"
CACHE_DIR = PROJECT_DIR / "results" / "cache" / "nf_sweep_v2_quickcheck"
for d in [QUICKCHECK_SAMPLE_ROOT, OUTPUT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 123
CHECKPOINT_EPOCH_FINAL = 99
REQUIRE_FINAL_CHECKPOINT_FOR_GENERATION = True
USE_PRODUCTION_SAMPLES_IF_QUICKCHECK_MISSING = False

MAX_RAW_REAL_CUBES = 16      # 16 raw cubes -> 512 2D slices with zthin=4
MAX_REAL_HIST = 512
MAX_REAL_PK = 512
PK_NBINS = 25

print("manifest:", MANIFEST_PATH, "exists=", MANIFEST_PATH.exists())
print("checkpoint_root:", CHECKPOINT_ROOT)
print("quickcheck samples:", QUICKCHECK_SAMPLE_ROOT)
print("production samples:", PRODUCTION_SAMPLE_ROOT)
print("output_dir:", OUTPUT_DIR)

## Discovery

This maps the 14 v2 runs to checkpoint status and existing quick-check samples. A run is considered ready for one-sample generation when its latest checkpoint is `checkpoint-epoch-0099`, unless `REQUIRE_FINAL_CHECKPOINT_FOR_GENERATION` is set to `False`.

In [ ]:
def latest_checkpoint(run_name: str) -> Path | None:
    ckpt_dir = CHECKPOINT_ROOT / f"{run_name}_checkpoints"
    if not ckpt_dir.exists():
        return None
    candidates = sorted(ckpt_dir.glob("checkpoint-epoch-*"))
    return candidates[-1] if candidates else None


def checkpoint_epoch(path: Path | None) -> int | None:
    if path is None:
        return None
    m = re.search(r"checkpoint-epoch-(\d+)$", path.name)
    return int(m.group(1)) if m else None


def quickcheck_sample_path(run_name: str) -> Path:
    return QUICKCHECK_SAMPLE_ROOT / f"{run_name}_seed{SEED}_raw_train_full_n1.npz"


def production_raw_sample_path(run_name: str) -> Path:
    return PRODUCTION_SAMPLE_ROOT / f"{run_name}_seed{SEED}_raw_train_full.npz"


def npz_shape(path: Path) -> tuple[int, ...] | None:
    if not path.exists():
        return None
    try:
        with np.load(path, allow_pickle=True) as data:
            key = "samples" if "samples" in data.files else data.files[0]
            return tuple(data[key].shape)
    except Exception:
        return None

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing {MANIFEST_PATH}. Run scripts/prepare_nf_sweep_v2_configs.py first.")

manifest = json.loads(MANIFEST_PATH.read_text())
rows = []
for idx, row in enumerate(manifest):
    run_name = row["run_name"]
    ckpt = latest_checkpoint(run_name)
    epoch = checkpoint_epoch(ckpt)
    qpath = quickcheck_sample_path(run_name)
    ppath = production_raw_sample_path(run_name)
    rows.append({
        "idx": idx,
        **row,
        "config_path": str(PROJECT_DIR / row["config"]),
        "checkpoint_path": str(ckpt) if ckpt else None,
        "checkpoint_epoch": epoch,
        "final_checkpoint": epoch == CHECKPOINT_EPOCH_FINAL,
        "quickcheck_sample": str(qpath),
        "has_quickcheck_sample": qpath.exists(),
        "quickcheck_shape": npz_shape(qpath),
        "has_production_raw_sample": ppath.exists(),
        "production_raw_shape": npz_shape(ppath),
    })
run_df = pd.DataFrame(rows)
display(run_df[[
    "idx", "run_name", "arch", "variant_tag", "checkpoint_epoch", "final_checkpoint",
    "has_quickcheck_sample", "quickcheck_shape", "has_production_raw_sample", "production_raw_shape",
]])

## Generate One Raw Sample Per Completed Model

Set `GENERATE_ONE_SAMPLE_PER_MODEL_NOW = True` and run this cell on a GPU node/session. It generates exactly one raw sample for each completed v2 model that does not already have a quick-check sample.

The outputs are intentionally separate from production samples:

`results/nf_sweep_v2/quickcheck_samples/*_n1.npz`

In [ ]:
GENERATE_ONE_SAMPLE_PER_MODEL_NOW = False
ONLY_MISSING_QUICKCHECK_SAMPLES = True
SMOKE_DEVICE = "cuda"
SMOKE_NUM_SAMPLES = 1
SMOKE_BATCH_SIZE = 1
SMOKE_NUM_STEPS = None  # None = full 500-step train scheduler. For faster smoke, set 25 with DPM below.
SMOKE_SCHEDULER = None  # e.g. "DPMSolverMultistepScheduler" when SMOKE_NUM_STEPS=25


def prepare_sampling_env() -> dict[str, str]:
    stub_root = CACHE_DIR / "python_stubs"
    (stub_root / "sklearn" / "metrics").mkdir(parents=True, exist_ok=True)
    (stub_root / "sklearn" / "__init__.py").write_text("from . import metrics\n")
    (stub_root / "sklearn" / "metrics" / "__init__.py").write_text(
        "def roc_curve(*args, **kwargs):\n"
        "    raise RuntimeError('sklearn.metrics.roc_curve is stubbed for cosmodiff sampling')\n"
    )
    env = os.environ.copy()
    env["COSMODIFF_DIR"] = str(COSMODIFF_DIR)
    env["COSMODIFF_STUB_SKLEARN"] = "1"
    env["TORCHDYNAMO_DISABLE"] = "1"
    env.pop("PYTHONNOUSERSITE", None)
    env["PYTHONPATH"] = f"{stub_root}:{COSMODIFF_DIR}:{PROJECT_DIR}:{env.get('PYTHONPATH', '')}"
    return env


if GENERATE_ONE_SAMPLE_PER_MODEL_NOW:
    venv_python = Path("/home/jiamingp/venvs/cosmodiff_nf/bin/python")
    python_bin = str(venv_python if venv_python.exists() else Path(sys.executable))
    env = prepare_sampling_env()
    rows_to_sample = []
    for row in run_df.to_dict("records"):
        output_path = Path(row["quickcheck_sample"])
        if ONLY_MISSING_QUICKCHECK_SAMPLES and output_path.exists():
            continue
        if REQUIRE_FINAL_CHECKPOINT_FOR_GENERATION and not row["final_checkpoint"]:
            continue
        if row["checkpoint_path"] is None:
            continue
        rows_to_sample.append(row)

    if not rows_to_sample:
        print("No runs need quick-check sampling. Check checkpoint completion and existing sample files above.")
    for row in rows_to_sample:
        run_name = row["run_name"]
        output_path = Path(row["quickcheck_sample"])
        output_path.parent.mkdir(parents=True, exist_ok=True)
        cmd = [
            python_bin,
            str(COSMODIFF_DIR / "scripts" / "cosmodiff_sample.py"),
            "--config", str(row["config_path"]),
            "--filepath", str(output_path),
            "--n_samples", str(SMOKE_NUM_SAMPLES),
            "--batch_size", str(SMOKE_BATCH_SIZE),
            "--seed", str(SEED),
            "--device", SMOKE_DEVICE,
            "--verbose",
        ]
        if SMOKE_SCHEDULER is not None:
            cmd += ["--scheduler", SMOKE_SCHEDULER]
        if SMOKE_NUM_STEPS is not None:
            cmd += ["--num_steps", str(SMOKE_NUM_STEPS)]
        print("\nGenerating one quick-check sample for", run_name)
        print("Output:", output_path)
        print("Command:", " ".join(shlex.quote(x) for x in cmd))
        subprocess.run(cmd, cwd=PROJECT_DIR, env=env, check=True)
    print("Done. Rerun Discovery and then Load Samples below.")
else:
    print("Set GENERATE_ONE_SAMPLE_PER_MODEL_NOW = True to generate one raw sample per completed v2 model.")

## Load Real And Generated Quick-Check Samples

This loads quick-check samples first. If `USE_PRODUCTION_SAMPLES_IF_QUICKCHECK_MISSING=True`, it can fall back to production raw samples.

In [ ]:
def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return np.array(arr, copy=True)
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return np.array(arr[idx], copy=True)


def load_npz_samples(path: Path) -> np.ndarray:
    with np.load(path, allow_pickle=True) as data:
        key = "samples" if "samples" in data.files else data.files[0]
        return as_nchw(np.asarray(data[key])).copy()


def selected_sample_path(row: dict[str, Any]) -> tuple[str | None, Path | None]:
    q = Path(row["quickcheck_sample"])
    if q.exists():
        return "quickcheck_raw_n1", q
    p = production_raw_sample_path(row["run_name"])
    if USE_PRODUCTION_SAMPLES_IF_QUICKCHECK_MISSING and p.exists():
        return "production_raw", p
    return None, None


def data_signature(config_path: Path) -> str:
    with config_path.open() as f:
        cfg = yaml.safe_load(f)
    data = cfg.get("data", {})
    keys = ["img_path", "reshape", "two_dim", "zthin", "n_samples", "seed", "log", "transform", "normalization", "norm_kwargs"]
    slim = {k: data.get(k) for k in keys if k in data}
    return json.dumps(slim, sort_keys=True, default=str)

real_cache: dict[str, np.ndarray] = {}
loaded: dict[str, dict[str, Any]] = {}
load_rows = []
for row in run_df.to_dict("records"):
    label, sample_path = selected_sample_path(row)
    if sample_path is None:
        load_rows.append({"run_name": row["run_name"], "loaded": False, "reason": "missing quickcheck sample"})
        continue
    config_path = Path(row["config_path"])
    sig = data_signature(config_path)
    if sig not in real_cache:
        real_cache[sig] = load_real_from_config(config_path, max_raw_samples=MAX_RAW_REAL_CUBES)
    real = real_cache[sig]
    generated = load_npz_samples(sample_path)
    loaded[row["run_name"]] = {"spec": row, "sample_label": label, "sample_path": sample_path, "real": real, "generated": generated}
    load_rows.append({
        "idx": row["idx"],
        "run_name": row["run_name"],
        "arch": row["arch"],
        "variant": row["variant_tag"],
        "loaded": True,
        "sample_label": label,
        "real_shape": tuple(real.shape),
        "generated_shape": tuple(generated.shape),
        "sample_path": str(sample_path),
    })
load_df = pd.DataFrame(load_rows)
display(load_df)
print("loaded runs:", len(loaded), "unique real references:", len(real_cache))

## One-Point Statistics

With one generated sample, this is a smoke test, not a final statistical result. It is still useful for detecting collapse to a near-constant field.

In [ ]:
def onepoint_summary(real: np.ndarray, generated: np.ndarray, bins: int = 120) -> dict[str, float]:
    rh = field_histogram(real, bins=bins)
    gh = field_histogram(generated, bins=bins)
    edges = np.asarray(rh["bin_edges"])
    width = float(np.mean(np.diff(edges)))
    hist_l1 = float(np.sum(np.abs(np.asarray(rh["hist"]) - np.asarray(gh["hist"]))) * width)
    return {
        "real_mean": rh["mean"],
        "generated_mean": gh["mean"],
        "real_std": rh["std"],
        "generated_std": gh["std"],
        "std_ratio": gh["std"] / max(rh["std"], 1e-30),
        "real_q01": rh["q01"],
        "generated_q01": gh["q01"],
        "real_q99": rh["q99"],
        "generated_q99": gh["q99"],
        "hist_l1": hist_l1,
    }

rows = []
for run_name, bundle in loaded.items():
    rows.append({
        "run_name": run_name,
        "arch": bundle["spec"]["arch"],
        "variant": bundle["spec"]["variant_tag"],
        "sample_label": bundle["sample_label"],
        "n_generated": len(bundle["generated"]),
        **onepoint_summary(evenly_limit(bundle["real"], MAX_REAL_HIST), bundle["generated"]),
    })
onepoint_df = pd.DataFrame(rows)
if len(onepoint_df):
    onepoint_df = onepoint_df.sort_values(["arch", "hist_l1"], na_position="last")
display(onepoint_df)
out = OUTPUT_DIR / "nf_sweep_v2_quickcheck_onepoint_metrics.csv"
onepoint_df.to_csv(out, index=False)
print("wrote", out)

In [ ]:
if loaded:
    for arch in sorted({b["spec"]["arch"] for b in loaded.values()}):
        items = [(k, v) for k, v in loaded.items() if v["spec"]["arch"] == arch]
        ncols = min(4, len(items))
        nrows = math.ceil(len(items) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows), squeeze=False)
        for ax in axes.ravel():
            ax.axis("off")
        for ax, (run_name, bundle) in zip(axes.ravel(), items):
            ax.axis("on")
            real_hist = field_histogram(evenly_limit(bundle["real"], MAX_REAL_HIST))
            gen_hist = field_histogram(bundle["generated"])
            edges = np.asarray(real_hist["bin_edges"])
            centers = 0.5 * (edges[:-1] + edges[1:])
            ax.plot(centers, real_hist["hist"], color="black", lw=2, label="real")
            ax.plot(centers, gen_hist["hist"], color="tab:blue", lw=1.8, label="generated")
            ax.set_yscale("log")
            ax.set_title(f"{bundle['spec']['variant_tag']}\n{bundle['sample_label']}, n={len(bundle['generated'])}", fontsize=9)
            ax.set_xlabel("normalized field value")
            ax.set_ylabel("density")
            ax.grid(alpha=0.25)
            ax.legend(fontsize=8)
        fig.suptitle(f"{arch}: quick-check one-point histograms")
        fig.tight_layout()
        out = OUTPUT_DIR / f"nf_sweep_v2_{arch}_quickcheck_onepoint_histograms.png"
        fig.savefig(out)
        print("wrote", out)

## Power Spectrum P(k)

For one generated sample, the blue generated P(k) is a single-sample curve. The black curve and gray band come from the real reference slices.

In [ ]:
pk_rows = []
pk_cache = {}
for run_name, bundle in loaded.items():
    real = evenly_limit(bundle["real"], MAX_REAL_PK)
    generated = bundle["generated"]
    pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
    pk_gen, _ = batch_power_spectra(generated, nbins=PK_NBINS)
    real_mean = np.nanmean(pk_real, axis=0)
    gen_mean = np.nanmean(pk_gen, axis=0)
    ratio = gen_mean / np.clip(real_mean, 1e-30, None)
    pk_cache[run_name] = {"kbins": kbins, "pk_real": pk_real, "pk_gen": pk_gen, "ratio": ratio}
    pk_rows.append({
        "run_name": run_name,
        "arch": bundle["spec"]["arch"],
        "variant": bundle["spec"]["variant_tag"],
        "sample_label": bundle["sample_label"],
        "n_generated": len(generated),
        **power_spectrum_summary(real, generated, nbins=PK_NBINS),
    })
pk_df = pd.DataFrame(pk_rows)
if len(pk_df):
    pk_df = pk_df.sort_values(["arch", "pk_log10_mae"], na_position="last")
display(pk_df)
out = OUTPUT_DIR / "nf_sweep_v2_quickcheck_pk_metrics.csv"
pk_df.to_csv(out, index=False)
print("wrote", out)

In [ ]:
if loaded:
    for arch in sorted({b["spec"]["arch"] for b in loaded.values()}):
        items = [(k, v) for k, v in loaded.items() if v["spec"]["arch"] == arch]
        ncols = min(4, len(items))
        nrows = math.ceil(len(items) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.4 * nrows), squeeze=False)
        for ax in axes.ravel():
            ax.axis("off")
        for ax, (run_name, bundle) in zip(axes.ravel(), items):
            ax.axis("on")
            info = pk_cache[run_name]
            kbins = info["kbins"]
            pk_real = info["pk_real"]
            pk_gen = info["pk_gen"]
            real_mean = np.nanmean(pk_real, axis=0)
            real_std = np.nanstd(pk_real, axis=0)
            gen_mean = np.nanmean(pk_gen, axis=0)
            ax.fill_between(kbins, np.clip(real_mean - real_std, 1e-30, None), real_mean + real_std, color="black", alpha=0.14)
            ax.plot(kbins, real_mean, color="black", lw=2, label="real")
            ax.plot(kbins, gen_mean, color="tab:blue", lw=1.8, marker="o", ms=2.5, label="generated")
            ax.set_yscale("log")
            ax.set_title(f"{bundle['spec']['variant_tag']}\n{bundle['sample_label']}, n={len(bundle['generated'])}", fontsize=9)
            ax.set_xlabel("k bin")
            ax.set_ylabel("P(k)")
            ax.grid(alpha=0.25)
            ax.legend(fontsize=8)
        fig.suptitle(f"{arch}: quick-check P(k)")
        fig.tight_layout()
        out = OUTPUT_DIR / f"nf_sweep_v2_{arch}_quickcheck_pk.png"
        fig.savefig(out)
        print("wrote", out)

## Generated Image Grid

This is the fastest visual check for blank/collapsed samples.

In [ ]:
if loaded:
    for arch in sorted({b["spec"]["arch"] for b in loaded.values()}):
        items = [(k, v) for k, v in loaded.items() if v["spec"]["arch"] == arch]
        ncols = min(7, len(items))
        nrows = math.ceil(len(items) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(2.0 * ncols, 2.25 * nrows), squeeze=False)
        for ax in axes.ravel():
            ax.axis("off")
        vals = np.concatenate([v["generated"].ravel() for _, v in items])
        vmin, vmax = np.nanpercentile(vals, [1, 99])
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = -1, 1
        for ax, (run_name, bundle) in zip(axes.ravel(), items):
            img = bundle["generated"][0, 0]
            ax.imshow(img, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
            ax.set_title(bundle["spec"]["variant_tag"], fontsize=9)
        fig.suptitle(f"{arch}: generated quick-check images")
        fig.tight_layout()
        out = OUTPUT_DIR / f"nf_sweep_v2_{arch}_quickcheck_images.png"
        fig.savefig(out)
        print("wrote", out)

## Training Status

This section reads checkpoint folders and, when available, latest metrics files.

In [ ]:
def metric_candidates(run_name: str) -> list[Path]:
    root = CHECKPOINT_ROOT / f"{run_name}_checkpoints"
    paths = []
    paths.extend(sorted(root.glob("metrics_epoch_*.json")))
    paths.extend(sorted(root.glob("metrics.json")))
    for ckpt in sorted(root.glob("checkpoint-epoch-*")):
        paths.extend(sorted(ckpt.glob("metrics*.json")))
    return paths

metric_rows = []
for row in run_df.to_dict("records"):
    paths = metric_candidates(row["run_name"])
    metrics = None
    if paths:
        with paths[-1].open() as f:
            metrics = json.load(f)
    epoch_loss = np.asarray((metrics or {}).get("epoch_loss", []), dtype=float)
    metric_rows.append({
        "idx": row["idx"],
        "run_name": row["run_name"],
        "arch": row["arch"],
        "variant": row["variant_tag"],
        "checkpoint_epoch": row["checkpoint_epoch"],
        "final_checkpoint": row["final_checkpoint"],
        "metrics_path": str(paths[-1]) if paths else None,
        "n_epochs_logged": len(epoch_loss),
        "latest_epoch_loss": float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
        "best_epoch_loss": float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
    })
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

In [ ]:
summary = run_df[["idx", "run_name", "arch", "variant_tag", "checkpoint_epoch", "final_checkpoint", "has_quickcheck_sample", "quickcheck_shape"]].rename(columns={"variant_tag": "variant"})
if "onepoint_df" in globals() and len(onepoint_df):
    summary = summary.merge(onepoint_df[["run_name", "n_generated", "generated_std", "std_ratio", "hist_l1"]], on="run_name", how="left")
if "pk_df" in globals() and len(pk_df):
    summary = summary.merge(pk_df[["run_name", "pk_log10_mae", "pk_ratio_low_k", "pk_ratio_mid_k", "pk_ratio_high_k"]], on="run_name", how="left")
summary = summary.sort_values(["arch", "hist_l1"], na_position="last")
display(summary)
out = OUTPUT_DIR / "nf_sweep_v2_quickcheck_summary.csv"
summary.to_csv(out, index=False)
print("wrote", out)